In [1]:
import os
import re
import cv2
import glob
import tempfile
import subprocess
import numpy as np
from math import log
from joblib import Parallel, delayed
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Path to your ripser executable
RIPSER_PATH = "./../../ripser/ripser" 

# Classes defined in your dataset
CLASSES = ['Normal', 'Pneumonia-Bacterial', 'Pneumonia-Viral', 'COVID-19', 'Tuberculosis', 'Emphysema']
DATASET_DIR = "dataset" # Fill in with the actual path

In [2]:
def image_to_point_cloud(image_path, max_points=200000):
    """Loads an image, applies Canny edge detection, and samples a point cloud."""
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    
    edges = cv2.Canny(img, 50, 150)
    points = np.argwhere(edges > 0)
    
    if len(points) < 10:
        return None
        
    if len(points) > max_points:
        idx = np.random.choice(len(points), max_points, replace=False)
        points = points[idx]
        
    return points

def parse_ripser_stdout(stdout_str):
    """Parses the standard output text from the Ripser executable."""
    intervals = {0: [], 1: []}
    current_dim = None
    
    for line in stdout_str.split('\n'):
        if "persistence intervals in dim" in line:
            match = re.search(r"dim (\d+):", line)
            if match:
                current_dim = int(match.group(1))
        elif current_dim in [0, 1] and line.strip().startswith('['):
            line_clean = line.replace('[', '').replace(')', '').replace(' ', '')
            parts = line_clean.split(',')
            if len(parts) == 2:
                try:
                    birth = float(parts[0])
                    death = float(parts[1]) if parts[1] != 'inf' else -1.0
                    intervals[current_dim].append((birth, death))
                except ValueError:
                    continue
    return intervals

def extract_betti_curves(intervals, num_bins=50, max_val=100.0):
    """Generates Betti curves as a fixed-length vector of 100 (50 for H0, 50 for H1)."""
    curves = []
    bins = np.linspace(0, max_val, num_bins)
    
    for dim in [0, 1]:
        dim_curve = np.zeros(num_bins)
        for birth, death in intervals[dim]:
            if death == -1.0: 
                death = max_val
            active = (bins >= birth) & (bins <= death)
            dim_curve[active] += 1
        curves.extend(dim_curve)
        
    return np.array(curves)

# def process_single_image_tda(image_path):
#     """Full TDA pipeline for a single image. Always returns a vector of length 100."""
#     default_features = np.zeros(100) 
    
#     points = image_to_point_cloud(image_path)
#     if points is None:
#         return default_features
        
#     with tempfile.NamedTemporaryFile(mode='w+', delete=False, suffix='.txt') as tmp_file:
#         for p in points:
#             tmp_file.write(f"{p[0]} {p[1]}\n")
#         tmp_file_path = tmp_file.name

#     try:
#         result = subprocess.run([RIPSER_PATH, tmp_file_path], capture_output=True, text=True, check=True)
#         intervals = parse_ripser_stdout(result.stdout)
#         features = extract_betti_curves(intervals)
#         return features
#     except Exception:
#         return default_features
#     finally:
#         if os.path.exists(tmp_file_path):
#             os.remove(tmp_file_path)

def process_single_image_raw(image_path, target_size=(32, 32)):
    """Loads an image, resizes it, and flattens it into a raw pixel vector."""
    default_raw = np.zeros(target_size[0] * target_size[1])
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return default_raw
    img_resized = cv2.resize(img, target_size)
    return img_resized.flatten() / 255.0

def calculate_persistent_entropy(intervals):
    """Calculates Persistent Entropy for H0 and H1."""
    features = []
    for dim in [0, 1]:
        lifespans = [d - b for b, d in intervals[dim] if d != -1.0]
        if not lifespans:
            features.extend([0.0] * 5)
            continue
        
        total_lifespan = sum(lifespans)
        # Probabilities
        probs = [l / total_lifespan for l in lifespans]
        entropy = -sum(p * log(p) for p in probs if p > 0)
        
        # Stats: Count, Min, Max, Mean, Entropy
        features.extend([len(lifespans), min(lifespans), max(lifespans), np.mean(lifespans), entropy])
    return np.array(features) # Returns 10 features

def extract_persistent_landscapes(intervals, num_bins=30, num_layers=2):
    """Simplified Persistent Landscape approximation."""
    landscape = np.zeros(num_bins * num_layers)
    bins = np.linspace(0, 100, num_bins)
    
    for dim in [0, 1]:
        # We process each layer
        for layer in range(num_layers):
            # For this simple implementation, we take the 'layer'-th largest lifespan
            # In a full TDA library (like 'gudhi'), this would be more precise
            lifespans = sorted([d - b for b, d in intervals[dim] if d != -1.0], reverse=True)
            
            if layer < len(lifespans):
                # Simple approximation: a triangle function centered at the midpoint
                # of the birth-death interval
                for b, d in intervals[dim]:
                    if d == -1: d = 100
                    mid = (b + d) / 2
                    width = (d - b) / 2
                    # Tent function
                    vals = np.maximum(0, width - np.abs(bins - mid))
                    landscape[layer*num_bins : (layer+1)*num_bins] += vals
                    
    return landscape # Returns 60 features (30 bins * 2 layers)

def process_single_image_tda(image_path):
    """Full TDA pipeline including Betti Curves, Entropy, and Landscapes."""
    # Total vector size: 100 (Betti) + 10 (Entropy) + 60 (Landscapes) = 170
    default_features = np.zeros(170)
    
    points = image_to_point_cloud(image_path)
    if points is None: return default_features
        
    with tempfile.NamedTemporaryFile(mode='w+', delete=False, suffix='.txt') as tmp_file:
        for p in points: tmp_file.write(f"{p[0]} {p[1]}\n")
        tmp_file_path = tmp_file.name

    try:
        result = subprocess.run([RIPSER_PATH, tmp_file_path], capture_output=True, text=True, check=True)
        intervals = parse_ripser_stdout(result.stdout)
        
        # Concatenate features
        betti = extract_betti_curves(intervals) # 100
        entropy = calculate_persistent_entropy(intervals) # 10
        landscapes = extract_persistent_landscapes(intervals) # 60
        
        return np.concatenate([betti, entropy, landscapes])
    except Exception:
        return default_features
    finally:
        if os.path.exists(tmp_file_path): os.remove(tmp_file_path)

In [3]:
def extract_features_for_split(split_name):
    """
    Extracts paths, labels, and features for a specific dataset split ('train', 'val', or 'test').
    """
    image_paths = []
    labels = []

    for class_idx, class_name in enumerate(CLASSES):
        # Construct path: e.g., dataset/train/Normal/*.png
        search_path = os.path.join(DATASET_DIR, split_name, class_name, "*.*") 
        class_paths = glob.glob(search_path)
        
        # Filter out non-image files just in case
        class_paths = [p for p in class_paths if p.lower().endswith(('.png', '.jpg', '.jpeg'))]
        
        image_paths.extend(class_paths)
        labels.extend([class_idx] * len(class_paths))

    print(f"[{split_name.upper()}] Found {len(image_paths)} images.")

    # Extract features in parallel
    X_tda = Parallel(n_jobs=-1)(delayed(process_single_image_tda)(path) for path in image_paths)
    X_raw = Parallel(n_jobs=-1)(delayed(process_single_image_raw)(path) for path in image_paths)

    return np.array(X_raw), np.array(X_tda), np.array(labels)

# Extract features for all three splits
print("Processing TRAIN set...")
X_train_raw, X_train_tda, y_train = extract_features_for_split('train')

print("\nProcessing VAL set...")
X_val_raw, X_val_tda, y_val = extract_features_for_split('val')

print("\nProcessing TEST set...")
X_test_raw, X_test_tda, y_test = extract_features_for_split('test')

Processing TRAIN set...
[TRAIN] Found 14551 images.

Processing VAL set...
[VAL] Found 1748 images.

Processing TEST set...
[TEST] Found 1737 images.


In [4]:
# --- MODEL 1: Pure Random Forest on Raw Pixels ---
print("\n=== TRAINING: Pure Random Forest (Raw Pixels) ===")
rf_raw = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)

# Fit on training data
rf_raw.fit(X_train_raw, y_train)

# Evaluate on test data
y_pred_raw = rf_raw.predict(X_test_raw)

print("Test Results for Raw Pixels:")
print(classification_report(y_test, y_pred_raw, target_names=CLASSES))


# --- MODEL 2: Random Forest on TDA Features ---
print("\n=== TRAINING: Random Forest + TDA (Betti Curves) ===")
rf_tda = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)

# Fit on training data
rf_tda.fit(X_train_tda, y_train)

# Evaluate on test data
y_pred_tda = rf_tda.predict(X_test_tda)

print("Test Results for TDA Features:")
print(classification_report(y_test, y_pred_tda, target_names=CLASSES))


=== TRAINING: Pure Random Forest (Raw Pixels) ===
Test Results for Raw Pixels:
                     precision    recall  f1-score   support

             Normal       0.88      0.93      0.90       300
Pneumonia-Bacterial       0.78      0.83      0.81       300
    Pneumonia-Viral       0.82      0.70      0.75       300
           COVID-19       0.83      0.88      0.85       300
       Tuberculosis       0.94      0.98      0.96       287
          Emphysema       0.85      0.78      0.81       250

           accuracy                           0.85      1737
          macro avg       0.85      0.85      0.85      1737
       weighted avg       0.85      0.85      0.85      1737


=== TRAINING: Random Forest + TDA (Betti Curves) ===
Test Results for TDA Features:
                     precision    recall  f1-score   support

             Normal       0.68      0.86      0.76       300
Pneumonia-Bacterial       0.58      0.64      0.61       300
    Pneumonia-Viral       0.58      0.

In [5]:
# Szybki test fuzji:
X_train_combined = np.hstack((X_train_raw, X_train_tda))
X_test_combined = np.hstack((X_test_raw, X_test_tda))

rf_combined = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf_combined.fit(X_train_combined, y_train)
y_pred_combined = rf_combined.predict(X_test_combined)

print("Wyniki po połączeniu Pixels + TDA:")
print(classification_report(y_test, y_pred_combined, target_names=CLASSES))

Wyniki po połączeniu Pixels + TDA:
                     precision    recall  f1-score   support

             Normal       0.87      0.92      0.89       300
Pneumonia-Bacterial       0.76      0.80      0.78       300
    Pneumonia-Viral       0.77      0.70      0.73       300
           COVID-19       0.84      0.85      0.85       300
       Tuberculosis       0.97      0.98      0.98       287
          Emphysema       0.85      0.80      0.82       250

           accuracy                           0.84      1737
          macro avg       0.84      0.84      0.84      1737
       weighted avg       0.84      0.84      0.84      1737



In [6]:
# Create combined features
X_train_combined = np.hstack((X_train_raw, X_train_tda))
X_test_combined = np.hstack((X_test_raw, X_test_tda))

# Train one strong model
print("Training Combined Model...")
rf_combined = RandomForestClassifier(n_estimators=300, max_depth=20, n_jobs=-1)
rf_combined.fit(X_train_combined, y_train)

y_pred_combined = rf_combined.predict(X_test_combined)
print(classification_report(y_test, y_pred_combined, target_names=CLASSES))

Training Combined Model...
                     precision    recall  f1-score   support

             Normal       0.87      0.93      0.90       300
Pneumonia-Bacterial       0.78      0.80      0.79       300
    Pneumonia-Viral       0.79      0.73      0.76       300
           COVID-19       0.87      0.86      0.86       300
       Tuberculosis       0.96      0.98      0.97       287
          Emphysema       0.85      0.82      0.84       250

           accuracy                           0.85      1737
          macro avg       0.85      0.85      0.85      1737
       weighted avg       0.85      0.85      0.85      1737



In [7]:
# --- MODEL 1: Pure Random Forest on Raw Pixels ---
print("\n=== TRAINING: Pure Random Forest (Raw Pixels) ===")
rf_raw = RandomForestClassifier(n_estimators=300, max_depth=20, random_state=42, n_jobs=-1)

# Fit on training data
rf_raw.fit(X_train_raw, y_train)

# Evaluate on test data
y_pred_raw = rf_raw.predict(X_test_raw)

print("Test Results for Raw Pixels:")
print(classification_report(y_test, y_pred_raw, target_names=CLASSES))


=== TRAINING: Pure Random Forest (Raw Pixels) ===
Test Results for Raw Pixels:
                     precision    recall  f1-score   support

             Normal       0.88      0.94      0.91       300
Pneumonia-Bacterial       0.80      0.82      0.81       300
    Pneumonia-Viral       0.83      0.73      0.78       300
           COVID-19       0.84      0.89      0.86       300
       Tuberculosis       0.95      0.98      0.96       287
          Emphysema       0.86      0.80      0.83       250

           accuracy                           0.86      1737
          macro avg       0.86      0.86      0.86      1737
       weighted avg       0.86      0.86      0.86      1737



In [8]:
# # Split into training and testing sets (80/20) maintaining class proportions
# X_train_raw, X_test_raw, X_train_tda, X_test_tda, y_train, y_test = train_test_split(
#     X_raw, X_tda, y, test_size=0.2, stratify=y, random_state=42
# )

# # --- MODEL 1: Pure Random Forest on Raw Pixels ---
# print("\n=== TRAINING: Pure Random Forest (Raw Pixels) ===")
# rf_raw = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
# rf_raw.fit(X_train_raw, y_train)
# y_pred_raw = rf_raw.predict(X_test_raw)

# print("Results for Raw Pixels:")
# print(classification_report(y_test, y_pred_raw, target_names=CLASSES))

# # --- MODEL 2: Random Forest on TDA Features ---
# print("\n=== TRAINING: Random Forest + TDA (Betti Curves) ===")
# rf_tda = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
# rf_tda.fit(X_train_tda, y_train)
# y_pred_tda = rf_tda.predict(X_test_tda)

# print("Results for TDA Features:")
# print(classification_report(y_test, y_pred_tda, target_names=CLASSES))